In [1]:
# Adapted from Parselmouth documentation at https://github.com/YannickJadoul/Parselmouth
import parselmouth
import praatio

import pandas as pd
import numpy as np
import os
import librosa
import glob
import sys
import synapseclient

sys.path.append(os.path.abspath('..'))

from extract_features_utils import clip_audio, get_jitter, get_shimmer, get_harmonic_to_noise_ratio, get_f0, get_formants
from pydub import AudioSegment

In [2]:
# Authenticate Synapse login credentials
syn = synapseclient.Synapse()
syn.login()

Welcome, sylviacho!



## Config

In [15]:
CSV_FILES_PATH = "../csv_files/misc/"                 
POSITIVE_FOLDER_PATH = "../audio_files/positive"
NEGATIVE_FOLDER_PATH = "../audio_files/negative"
TIME_DATA_PARQUET_PATH = "../parquet_files/"

os.makedirs(POSITIVE_FOLDER_PATH, exist_ok=True)
os.makedirs(NEGATIVE_FOLDER_PATH, exist_ok=True)
os.makedirs(TIME_DATA_PARQUET_PATH, exist_ok=True)

## Helper functions

In [4]:
# Helper to skip files shorter than 0.1s
def is_valid_audio(file_path):
    try:
        audio = AudioSegment.from_file(file_path, format="m4a")
        return len(audio) > 100
    except:
        return False

# Returns file path to converted .wav file
def convert_to_wav(input_file, output_dir):
    if not is_valid_audio(input_file):
        print(f"Skipping invalid or empty file: {input_file}")
        return None

    base_name = os.path.splitext(os.path.basename(input_file))[0]
    output_path = os.path.join(output_dir, f"{base_name}.wav")
    try:
        audio = AudioSegment.from_file(input_file, format="m4a")
    except Exception:
        try:
            audio = AudioSegment.from_file(input_file, format="mp4")
        except Exception as e:
            print(f"Conversion failed for {input_file}: {e}")
            return None

    try:
        audio.export(output_path, format="wav")
        return output_path
    except Exception as e:
        print(f"Export failed for {input_file}: {e}")
        return None

def convert_folder_to_wav(input_dir):
    for filename in os.listdir(input_dir):
        if filename.endswith(".m4a"):
            input_path = os.path.join(input_dir, filename)
            health_code = os.path.splitext(filename)[0]
            wav_path = os.path.join(input_dir, f"{health_code}.wav")
            if os.path.exists(wav_path):
                continue
            result = convert_to_wav(input_path, input_dir)

# Adapted from https://github.com/Sage-Bionetworks/mPower-sdata/blob/master/examples/mPower-bootstrap.py
# Downloads ALL matching audio files, customize batch size accordingly.
def extract_audio_files(output_dir, diagnosis=False, batch_size=100):
    os.makedirs(output_dir, exist_ok=True)

    # Load and filter survey data
    survey_data = pd.read_csv(CSV_FILES_PATH + "survey_data.csv")
    survey_data = survey_data[survey_data["professional-diagnosis"] == diagnosis]
    all_healthcodes = survey_data["healthCode"].tolist()

    all_mappings = []

    for i in range(0, len(all_healthcodes), batch_size):
        batch = all_healthcodes[i:i + batch_size]
        healthcodes_str = "','".join(batch)
        
        query = f"SELECT * FROM syn5511444 WHERE healthCode IN ('{healthcodes_str}')"
        subset_query = syn.tableQuery(query)
        subset_df = subset_query.asDataFrame()
        subset_df["audio_audio.m4a"] = subset_df["audio_audio.m4a"].astype(str)
        file_map = syn.downloadTableColumns(subset_query, "audio_audio.m4a")

        for file_handle_id, m4a_path in file_map.items():
            wav_path = convert_to_wav(m4a_path, output_dir)
            matched_row = subset_df[subset_df["audio_audio.m4a"] == file_handle_id]
            if not matched_row.empty:
                healthcode = matched_row.iloc[0]["healthCode"]
                all_mappings.append({"healthCode": healthcode, "wav_path": wav_path})
            else:
                print(f"File handle ID {file_handle_id} not found in subset_df")

    return pd.DataFrame(all_mappings)

# Return dataframe (s) that contain temporal data as a singular parquet file
# Extract formant values from folder containing .wav files
def get_formants_over_time_parquet(folder_path, df, output_parquet_path):
    wav_files = glob.glob(os.path.join(folder_path, "*.wav"))
    all_time_series = []

    for wav in wav_files:
        try:
            # Match .wav file to its current row in the DataFrame
            row = df.loc[df["wav_path"] == wav]
            if row.empty:
                print(f"No metadata found for: {wav}")
                continue

            # Get healthCode and gender info
            health_code = row["healthCode"].values[0]
            gender_str = row["gender"].values[0].lower() if "gender" in row and pd.notna(row["gender"].values[0]) else "unknown"
            gender_flag = 1 if gender_str == "female" else 0
            max_formant = 5500 if gender_flag == 1 else 5000

            # Load and trim audio
            audio, sr = librosa.load(wav, sr=None)
            trimmed_audio, idx = clip_audio(audio)
            sound = parselmouth.Sound(trimmed_audio, sampling_frequency=sr)

            # Get F0 over time
            pitch = sound.to_pitch()
            f0_times = [pitch.get_time_from_frame_number(i) for i in range(1, pitch.get_number_of_frames() + 1)]
            f0_values = [pitch.get_value_in_frame(i) for i in range(1, pitch.get_number_of_frames() + 1)]

            # Get formants over time
            time_step = 0.01
            formants = sound.to_formant_burg(time_step, 5, max_formant, 0.025, 50)
            f1, f2, f3 = [], [], []
            for t in f0_times:
                f1.append(formants.get_value_at_time(1, t))
                f2.append(formants.get_value_at_time(2, t))
                f3.append(formants.get_value_at_time(3, t))

            # Create and store DataFrame
            time_df = pd.DataFrame({
                "time": f0_times,
                "F0": f0_values,
                "F1": f1,
                "F2": f2,
                "F3": f3,
                "healthCode": health_code
            })

            all_time_series.append(time_df)

        except Exception as e:
            print(f"Error processing {wav}: {e}")
            continue

    # Combine all and write to parquet
    if all_time_series:
        full_df = pd.concat(all_time_series, ignore_index=True)
        full_df.to_parquet(output_parquet_path, index=False)
        print(f"Saved all data to: {output_parquet_path}")
    else:
        print("No valid data to save.")

# Data extraction

In [5]:
# Run this cell if you want to re-extract the data. You'll need around ~100GB of free space on your computer
df_pos = extract_audio_files(POSITIVE_FOLDER_PATH, diagnosis=True)
df_neg = extract_audio_files(NEGATIVE_FOLDER_PATH, diagnosis=False)

survey = pd.read_csv(os.path.join(CSV_FILES_PATH, "survey_data.csv"))
survey = survey.drop_duplicates(subset="healthCode", keep="first")

df_pos = df_pos.merge(survey[["healthCode","gender"]], on="healthCode", how="left")
df_neg = df_neg.merge(survey[["healthCode","gender"]], on="healthCode", how="left")

pos_parquet = os.path.join(TIME_DATA_PARQUET_PATH, "positive.parquet")
neg_parquet = os.path.join(TIME_DATA_PARQUET_PATH, "negative.parquet")

get_formants_over_time_parquet(POSITIVE_FOLDER_PATH, df_pos, pos_parquet)
get_formants_over_time_parquet(NEGATIVE_FOLDER_PATH, df_neg, neg_parquet)

print("Done.\nSaved:")
print("  ", pos_parquet)
print("  ", neg_parquet)

Querying table/view: 'syn5511444' ...:   0%|         | 0.00/100 [00:01<?, ?it/s]

[syn5511444]: Downloaded to /Users/kyledy/.synapseCache/254/161260254/SYNAPSE_TABLE_QUERY_161260254.csv


Skipping invalid or empty file: /Users/kyledy/.synapseCache/341/5811341/audio_audio.m4a-32ce4a7f-e98f-420a-a175-771f8940a23e4664184630692169645.tmp
Skipping invalid or empty file: /Users/kyledy/.synapseCache/268/6049268/audio_audio.m4a-32464813-9ecf-45a0-92fd-40c10dd5dadb8887105179483454413.tmp
Skipping invalid or empty file: /Users/kyledy/.synapseCache/808/5522808/audio_audio.m4a-da63fae4-ad65-4338-a246-7beca916772a1105219034040941526.tmp
Skipping invalid or empty file: /Users/kyledy/.synapseCache/379/5446379/audio_audio.m4a-2fc494a9-b60e-43a9-8866-edf561500f506221761593183990643.tmp
Skipping invalid or empty file: /Users/kyledy/.synapseCache/988/5662988/audio_audio.m4a-6315fd01-f2e1-4a90-b2d2-c765bd1db2b78161460049553722071.tmp
Skipping invalid or empty file: /Users/kyledy/.synapseCache/720/5658720/audio_audio.m4a-98c29a9b-3db1-4aa5-bcff-ee9032994c3d9048553270211683704.tmp
Skipping invalid or empty file: /Users/kyledy/.synapseCache/520/5913520/audio_audio.m4a-7ab883e8-6446-4175-85cd-

Create CSV FileHandle: 100%|██████████████| 10.4k/10.4k [00:01<00:00, 6.71kit/s]

[syn5511444]: Downloaded to /Users/kyledy/.synapseCache/85/161261085/SYNAPSE_TABLE_QUERY_161261085.csv


...

[syn5511444]: Downloaded to /Users/kyledy/.synapseCache/644/161279644/SYNAPSE_TABLE_QUERY_161279644.csv


Skipping invalid or empty file: /Users/kyledy/.synapseCache/322/5552322/audio_audio.m4a-2a98f61c-d97d-4932-b45e-5cc66d83f92f8451728690982667430.tmp


Create CSV FileHandle: 100%|████████████████████| 360/360 [00:01<00:00, 346it/s]

[syn5511444]: Downloaded to /Users/kyledy/.synapseCache/705/161279705/SYNAPSE_TABLE_QUERY_161279705.csv


Saved all data to: ../full_data_parquet/positive.parquet
Saved all data to: ../full_data_parquet/negative.parquet
Done.
Saved:
   ../full_data_parquet/positive.parquet
   ../full_data_parquet/negative.parquet


In [17]:
df = pd.read_parquet(os.path.join(TIME_DATA_PARQUET_PATH, "negative.parquet"))
print(df.shape)

(23433247, 6)
